# Compile the bird YOLO for the Raspberry Pi AI HAT+ 2 (Hailo-10H)

Turns `bird_yolo11n_v1.pt` into `bird_yolo11n_v1.hef`, quantized to INT8 with motion-view calibration crops.
Everything is read from and written to **Google Drive `MyDrive/quant/`**, so a Colab reset loses nothing that was already saved.

**Which model** (set in the first cell):
- `MODEL = "bird_yolo11n_v1_gray_stack"`, `CALIBRATION = "calibration_gray_stack.zip"`: the model builds the motion view itself from 3 gray crops. Upload `bird_yolo11n_v1_gray_stack.onnx` (built on the Mac by `vision/deploy/motion_view_network.py`).
- `MODEL = "bird_yolo11n_v1"`, `CALIBRATION = "calibration_crops.zip"`: the original model fed the CPU's motion view.
- `NMS_ENGINE = "nn_core"` tries duplicate-box removal on the chip; `"cpu"` keeps it in HailoRT on the Pi.

**Before running**
1. Runtime → Change runtime type → **T4 GPU**.
2. In Google Drive, folder `quant/` must hold:
   - the Hailo Dataflow Compiler wheel (`hailo_dataflow_compiler-5.4.0-py3-none-linux_x86_64.whl`)
   - `bird_yolo11n_v1.pt`
   - `calibration_crops.zip` (from `vision/deploy/export_calibration_crops.py`)
   - the latest `compile_hef.py` (from `vision/deploy/`)
3. On the Mac, run `caffeinate -dimsu` in a Terminal and keep this tab open.
4. Run all cells in order. **After a reset, just run them all again**: finished steps are skipped.

In [ ]:
FOLDER = "/content/drive/MyDrive/quant"
MODEL = "bird_yolo11n_v1_gray_stack"
CALIBRATION = "calibration_gray_stack.zip"
NMS_ENGINE = "cpu"
CALIBRATION_COUNT = 256
from google.colab import drive
drive.mount("/content/drive")
%cd {FOLDER}
!uname -m
!nvidia-smi -L || echo "no GPU: optimization will be slower"
!ls -la {FOLDER}

Installs the compiler in its own Python 3.12 environment (DFC 5.4.0 needs TensorFlow 2.19.1, which has no Python 3.13 build). The environment lives in `/content/dfc`, so it is rebuilt after every reset (3–6 minutes).

In [ ]:
PYTHON_FROM_DOWNLOAD_PAGE = "3.12"
import glob, re, subprocess, sys, zipfile
wheels = glob.glob(f"{FOLDER}/hailo_dataflow_compiler*.whl")
assert wheels, f"put the Hailo Dataflow Compiler .whl in {FOLDER} first"
subprocess.run(["cp", wheels[0], "/content/"], check=True)
wheel = "/content/" + wheels[0].split("/")[-1]
with zipfile.ZipFile(wheel) as archive:
    metadata = archive.read(next(n for n in archive.namelist() if n.endswith(".dist-info/METADATA"))).decode()
requires = re.search(r"^Requires-Python: (.+)$", metadata, re.M)
tag = re.search(r"-cp(\d)(\d+)-", wheel)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv", "packaging"], check=True)
from packaging.specifiers import SpecifierSet
allowed = SpecifierSet(requires.group(1)) if requires else SpecifierSet()
candidates = [f"{tag.group(1)}.{tag.group(2)}"] if tag else ["3.10", "3.11", "3.12"]
python_version = PYTHON_FROM_DOWNLOAD_PAGE or next((v for v in candidates if allowed.contains(f"{v}.0")), None)
assert python_version, f"wheel needs Python {requires.group(1) if requires else '?'}; none of {candidates} fit"
print("wheel:", wheel, "| Requires-Python:", requires.group(1) if requires else "not stated", "| using Python", python_version)
subprocess.run("apt-get -qq install -y graphviz graphviz-dev", shell=True)
subprocess.run(f"uv python install {python_version}", shell=True, check=True)
subprocess.run(f"uv venv --clear --python {python_version} /content/dfc", shell=True, check=True)
installed = subprocess.run(f"uv pip install --python /content/dfc/bin/python '{wheel}'", shell=True, capture_output=True, text=True)
print(installed.stdout[-3000:], installed.stderr[-6000:])
assert installed.returncode == 0, "installing the compiler failed: the lines above say why"
subprocess.run("uv pip install --python /content/dfc/bin/python onnx", shell=True)
subprocess.run(["/content/dfc/bin/python", "-c", "import hailo_sdk_client; print('hailo_sdk_client', hailo_sdk_client.__version__)"], check=True)

Export to ONNX at 640 in Colab's own Python (only for `bird_yolo11n_v1`; the gray-stack model is built on the Mac and uploaded). Skipped if the ONNX file is already in Drive.

In [ ]:
import os
onnx_path = f"{FOLDER}/{MODEL}.onnx"
if os.path.exists(onnx_path):
    print("already there:", onnx_path)
elif MODEL == "bird_yolo11n_v1":
    !pip install -q ultralytics onnx onnxslim
    from ultralytics import YOLO
    print(YOLO(f"{FOLDER}/bird_yolo11n_v1.pt").export(format="onnx", imgsz=640, opset=11, simplify=True, dynamic=False))
else:
    raise SystemExit(f"upload {MODEL}.onnx to {FOLDER} first (vision/deploy/motion_view_network.py on the Mac)")

Compile in two separate runs: (1) INT8 optimization, saved as `bird_yolo11n_v1_optimized.har`; (2) compiling that file for the chip in a fresh process, so the optimization's memory is freed first. Finished steps are skipped. Full logs go to `optimize.log` and `compile.log` in `quant/`.

**Exit code 0** = success. **137** = Colab ran out of memory and killed it. Anything else: send the log's last lines. Done at `saved bird_yolo11n_v1.hef`.

In [ ]:
import os
os.chdir(FOLDER)
hef_path = f"{FOLDER}/{MODEL}.hef"
optimized_path = f"{FOLDER}/{MODEL}_optimized.har"
if not os.path.exists(optimized_path):
    print("step 1 of 2: INT8 optimization (about 15 minutes)")
    !/content/dfc/bin/python compile_hef.py --optimize-only --onnx {MODEL}.onnx --calibration {CALIBRATION} --calibration-count {CALIBRATION_COUNT} --nms-engine {NMS_ENGINE} --hw-arch hailo10h --output {MODEL}.hef > optimize_{MODEL}.log 2>&1; echo "optimize exit code: $?"
    !grep -E "float:|int8:|saved|rror" optimize_{MODEL}.log | tail -20
if os.path.exists(hef_path) and os.path.getsize(hef_path) > 0:
    print("already compiled:", hef_path)
elif os.path.exists(optimized_path):
    print("step 2 of 2: compile for the Hailo-10H from", optimized_path, "(about 40 minutes, may print nothing for a long time)")
    !/content/dfc/bin/python compile_hef.py --resume --hw-arch hailo10h --output {MODEL}.hef > compile_{MODEL}.log 2>&1; echo "compile exit code: $?"
    !tail -25 compile_{MODEL}.log
    !free -h

Check the result: the `.hef` should be several MB, not 0. Then download it from Google Drive `quant/` into `models/` on the Mac and run `vision/deploy/push_to_pi.sh`.

In [ ]:
!ls -la {FOLDER}/*.hef {FOLDER}/*.har